In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, fixed


# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d, build_tissue
from render import render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap, NormalizeData
from optics import kryostat, psf_project, mask_collapse, detector
from parameter import (P, CellGeometry, MarkerPanel, PanelMarker, CellType, Detector,
                       dapi_marker,
                       BlobNoise, ClusterNoise, NetworkNoise, FibreNoise, SheetNoise,
                       Optics, TissueGeometry)
from artifacts import build_artifacts
import config as cfg


%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED       = cfg.SEED
N_CAND     = cfg.N_CAND
SIZE       = cfg.SIZE
TILE       = cfg.TILE
K          = cfg.K
L_MIN, L   = cfg.L_MIN, cfg.L
UM_PER_VOX = cfg.UM_PER_VOX
Z_RATIO    = cfg.Z_RATIO
SPACING    = cfg.SPACING
VOL        = cfg.CELL_VOL
TISSUE_VOL = cfg.TISSUE_VOL
N_POOLS    = cfg.N_POOLS


GEOM = CellGeometry()
DETECTOR = Detector()
FRACTIONS = [0.7, 0.3]

# ONE POOL PER (marker, component) PAIR, so two markers using the same noise kind still draw
# independent frozen fields. 3 markers x 3 components = 9.
N_POOLS = 12

# 4) DRAW THE TAPE
tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, GEOM, size=VOL, spacing=SPACING, i=0, L=L, l_min=L_MIN)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, geom = GEOM)
fig = plot_surface_xyz_html(cell = cell, geom = GEOM,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:

PANEL = MarkerPanel(
    Markers = dict(
        DAPI = dapi_marker(),


        r = PanelMarker(name="r", fluorophore="APC", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.80, s=1.40, mu= .55, width=.45, sharp=5.0,
                             scale=.35, clust=2.00, fill=.22, soft=.25),
                FibreNoise  (w=.30, s=1.30, mu= .20, width=.70, sharp=4.0,
                             lam=.25, length=6.0),
                NetworkNoise(w=.40, s=1.30, mu= .50, width=.55, sharp=3.5,
                             scale=.80, coherence=.40),
            ], artifact_affinity=1.2),

        g = PanelMarker(name="g", fluorophore="FITC", amp=2.0, polarity=0.2,
            noise_components=[
                BlobNoise   (w=.50, s=1.50, mu= .10, width=.60, sharp=7.5,
                             scale=.45),
                SheetNoise  (w=.60, s=1.50, mu= .40, width=1.20, sharp=6.0,
                             lam=1.90, coherence=.35, length=6.0),
                NetworkNoise(w=.90, s=1.40, mu= .20, width=1.10, sharp=7.0,
                             scale=1.00, coherence=.60),
            ], artifact_affinity=0.2)
    )
)


IMG = (VOL[1], VOL[2], len(PANEL.names))
tape.drawSensor(shape=IMG)

# edge_softness=0 -> clipped EXACTLY at the plasma membrane. All the softness in the final
# image is made by psf_project and detector, not baked into the object.
# For this test, express at level 1 for every marker
SOLO = CellType(name="solo", Geometry=GEOM,
                Expression={k: P(0., 2., .05, 1.0, f"expr {k}") for k in PANEL.names})
img = render_image(tape, PANEL, cell, spacing=SPACING, geom=GEOM,
                   um_per_vox=UM_PER_VOX, edge_softness=0.0,
                   expression={k: SOLO.level(k) for k in PANEL.names})

rgb = to_rgb(cell["cell"], img)

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = GEOM.ELONG.v,
                        polar_deg = GEOM.POLAR_DEG.v, azim_deg = GEOM.AZIM_DEG.v, roll_deg = GEOM.ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)


subs = {marker: kryostat(v, optics) for marker, v in img.items()}   # keep BOTH vol and z

img_psf = np.stack([psf_project(v, z, optics, PANEL.Markers[name].fluorophore)
                    for name, (v, z) in subs.items()], -1)

In [ ]:
sub, sub_z = kryostat(cell["cell"], optics)
sub_n, sub_n_z = kryostat(cell["nuc"], optics)


plt.imshow(NormalizeData(img_psf))
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
# Pre-normalize
img_psf_norm = (img_psf - np.min(img_psf)) / (np.max(img_psf) - np.min(img_psf))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

### 1.2.2) 2d Projection + Detector Model

In [ ]:
markers = list(subs.keys())

img_adu = detector(img_psf, PANEL, DETECTOR, optics, tape, markers)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(NormalizeData(img_psf), origin="lower")
ax[0].set_title("clean projection (object x PSF)")
ax[1].imshow(NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0)), origin="lower")
ax[1].set_title("detector frame (AF + shot + read)")
ax[2].imshow(NormalizeData(img_adu[..., 0]), cmap="magma", origin="lower")
ax[2].set_title(f"channel 0 ({markers[0]}, {PANEL.Markers[markers[0]].fluorophore}) in ADU")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Pre-normalize
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

# 2) Synthetic tissue generation

## 2) Tissue directionality

In [ ]:
_NOISE_CAL = 0.85
IDX6 = ((0, 0), (1, 1), (2, 2), (0, 1), (0, 2), (1, 2))
_S2 = 1.0 / np.sqrt(2.0)
_S6 = 1.0 / np.sqrt(6.0)
E5 = np.array([
    [-_S6, -_S6, 2.0 * _S6, 0.0, 0.0, 0.0],
    [_S2, -_S2, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, _S2, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, _S2, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, _S2],
])

def unit(v, axis=0):
    """v / |v|, safe at zero."""
    v = np.asarray(v, np.float64)
    return v / np.maximum(np.linalg.norm(v, axis=axis, keepdims=True), 1e-12)

def to33(q6):
    """(6, ...) -> (..., 3, 3), the layout numpy.linalg.eigh wants."""
    q = np.asarray(q6, np.float64)
    A = np.empty(q.shape[1:] + (3, 3), np.float64)
    for k, (i, j) in enumerate(IDX6):
        A[..., i, j] = q[k]
        A[..., j, i] = q[k]
    return A

def frob(q6):
    """The Frobenius norm squared, sum_ij Q_ij^2."""
    q = np.asarray(q6)
    return q[0] ** 2 + q[1] ** 2 + q[2] ** 2 + 2.0 * (q[3] ** 2 + q[4] ** 2 + q[5] ** 2)

# Q = w (n(x)n - I/3), stored as its 6 independent components (xx,yy,zz,xy,xz,yz)
def uni6(n, w):
    """A director as a Q-tensor, in 6-component storage: Q = w (n (x) n - I/3)"""
    n = unit(n)
    w = np.asarray(w, np.float64)
    return np.stack([w * (n[i] * n[j] - (1.0 / 3.0 if i == j else 0.0)) for i, j in IDX6])


def decompose(q6):
    """Q -> (director (3, ...), eigenvalue gap (...))."""
    # ascending
    lam, vec = np.linalg.eigh(to33(q6))          
    return np.moveaxis(vec[..., -1], -1, 0), lam[..., -1] - lam[..., -2]

def coherence(gap): 
    """gap -> S in [0, 1). """
    return gap / (1.0 + gap)


def noise6(c, w):
    """An ISOTROPIC random Q-tensor from five iid unit-variance fields, plus its own weight map."""
    c = np.asarray(c, np.float64)
    q = np.einsum("kc,k...->c...", E5, c)
    wn = float(w) * _NOISE_CAL
    return wn * q, wn * np.sqrt(1.5 * frob(q))

def slerp_axis(d, n, a):
    """Rotate the axis `d` a fraction `a` of the way toward the axis `n`, along the shortest arc.
    """
    d = unit(np.asarray(d, np.float64))
    n = unit(np.asarray(n, np.float64))
    a = float(a)
    if a <= 0.0:
        return d
    # the near end of the headless axis
    if float(d @ n) < 0.0:
        n = -n                                   
    g = float(np.arccos(np.clip(float(d @ n), -1.0, 1.0)))
    if g <= 1e-12:
        return d
    k = unit(np.cross(d, n))
    t = a * g
    return (d * np.cos(t) + np.cross(k, d) * np.sin(t) + k * float(k @ d) * (1.0 - np.cos(t)))


### PRESSURE PACK
from functools import reduce
from math import gcd, gamma
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra
import heapq


def metric_axes(aspect, ndim):
    """Determinant-normalised half-axes, so vol{d <= w} is the same at any aspect."""
    n = float(ndim)
    ra = float(aspect) ** ((n - 1.0) / n)
    return ra, float(aspect) ** (-1.0 / n)


def distance_uniform(shape, seeds, axis, aspect):
    """Closed form for a constant director. No graph, no sweeps, no error.

        d(x) = sqrt( (e.u / r_a)^2 + |e - (e.u)u|^2 / r_b^2 ),   e = x - seed

    `seeds` are index tuples in array order; `axis` is in the same order.
    """
    ndim = len(shape)
    ra, rb = metric_axes(aspect, ndim)
    u = np.asarray(axis, np.float32)
    u = u / max(float(np.linalg.norm(u)), 1e-12)
    grid = np.stack(np.mgrid[tuple(slice(0, s) for s in shape)]).astype(np.float32)
    d = np.full(shape, np.inf, np.float32)
    for s in seeds:
        e = grid - np.asarray(s, np.float32).reshape((ndim,) + (1,) * ndim)
        par = np.einsum("i,i...->...", u, e)
        per2 = np.maximum((e * e).sum(0) - par * par, 0.0)
        np.minimum(d, np.sqrt((par / ra) ** 2 + per2 / (rb * rb)), out=d)
    return d

def stencil(ndim, aspect):
    """Neighbour offsets for the geodesic graph: every PRIMITIVE integer offset within a
    radius. """
    r = 3 if (ndim == 2 and aspect >= 1.6) else 2
    out = []
    for d in np.ndindex(*(2 * r + 1,) * ndim):
        v = tuple(int(x) - r for x in d)
        if all(x == 0 for x in v):
            continue
        if reduce(gcd, [abs(x) for x in v]) == 1:      # primitive only
            out.append(v)
    return out

def anisotropic_graph(director, aspect, st=None):
    """Sparse neighbour graph whose edge weights follow the local director.

    Edge p->q is priced as the MEAN of the step cost at p and at q. That makes the graph
    symmetric, so one undirected Dijkstra gives an exact lattice geodesic, and multi-source
    is free (`min_only=True` returns the minimum over all seeds in a single pass).
    """
    ndim = director.shape[0]
    shape = director.shape[1:]
    st = stencil(ndim, aspect) if st is None else st
    ra, rb = metric_axes(aspect, ndim)
    n = int(np.prod(shape))
    idx = np.arange(n).reshape(shape)
    rows, cols, data = [], [], []
    for off in st:
        e = np.asarray(off, np.float32)
        par = np.einsum("i,i...->...", e, director)
        per2 = np.maximum(float(e @ e) - par * par, 0.0)
        cost = np.sqrt((par / ra) ** 2 + per2 / (rb * rb)).astype(np.float32)
        src = tuple(slice(max(0, -o), s - max(0, o)) for o, s in zip(off, shape))
        dst = tuple(slice(max(0, -o) + o, s - max(0, o) + o) for o, s in zip(off, shape))
        if any(sl.stop <= sl.start for sl in src):
            continue
        rows.append(idx[src].ravel())
        cols.append(idx[dst].ravel())
        data.append((0.5 * (cost[src] + cost[dst])).ravel())
    g = coo_matrix((np.concatenate(data).astype(np.float64),
                    (np.concatenate(rows), np.concatenate(cols))), shape=(n, n))
    return g.tocsr()

def distance_geodesic(graph, seeds, shape):
    """Exact multi-source lattice geodesic; d = min over this structure's own seeds."""
    flat = [int(np.ravel_multi_index(s, shape)) for s in seeds]
    d = dijkstra(graph, directed=False, indices=flat, min_only=True)
    return d.reshape(shape).astype(np.float32)

def distance_field(shape, seeds, aspect, director=None, axis=None, downsample=2,
                   graph=None):
    """The distance field for one structure, by whichever route is right."""
    if director is None:
        return distance_uniform(shape, seeds, axis, aspect)
    f = max(int(downsample), 1)
    if f == 1:
        g = anisotropic_graph(director, aspect) if graph is None else graph
        return distance_geodesic(g, seeds, shape)
    small = tuple(max(s // f, 2) for s in shape)
    u = np.stack([ndi.zoom(director[a], [t / s for t, s in zip(small, shape)], order=1,
                           mode="nearest") for a in range(director.shape[0])])
    u = u / np.maximum(np.linalg.norm(u, axis=0, keepdims=True), 1e-12)
    sd = [tuple(int(np.clip(c * t // s, 0, t - 1)) for c, t, s in zip(p, small, shape))
          for p in seeds]
    g = anisotropic_graph(u.astype(np.float32), aspect) if graph is None else graph
    d = distance_geodesic(g, sd, small)
    up = ndi.zoom(d, [s / t for s, t in zip(shape, small)], order=1, mode="nearest",
                  grid_mode=True)
    return (up * float(f)).astype(np.float32)


def stencil(ndim, aspect):
    """Neighbour offsets for the geodesic graph"""
    r = 3 if (ndim == 2 and aspect >= 1.6) else 2
    out = []
    for d in np.ndindex(*(2 * r + 1,) * ndim):
        v = tuple(int(x) - r for x in d)
        if all(x == 0 for x in v):
            continue
        if reduce(gcd, [abs(x) for x in v]) == 1:      # primitive only
            out.append(v)
    return out

def place_seeds(shape, counts, draws, spread=1.0):
    """Best-candidate ("Mitchell") sampling: each new seed is the FARTHEST of a batch of
    frozen candidates from every seed already placed. 
    This is a selction from frozen draws"""
    shape = np.asarray(shape, float)
    ndim = len(shape)
    cand = (0.5 + (np.asarray(draws, float)[:, :ndim] - 0.5) * float(spread)) * shape
    cand = np.clip(np.rint(cand), 0, shape - 1).astype(int)
    out, placed, cur = [], [], 0
    for k in counts:
        per = []
        for _ in range(int(k)):
            batch = cand[cur:cur + 16]
            cur += 16
            if batch.size == 0:
                break
            if placed:
                d2 = ((batch[:, None, :] - np.asarray(placed)[None]) ** 2).sum(-1).min(1)
                p = batch[int(np.argmax(d2))]
            else:
                p = batch[0]
            placed.append(tuple(p))
            per.append(tuple(int(v) for v in p))
        out.append(per)
    return out

def _prune_to_seeds(labels, seeds, conn):
    """Release any claimed voxel not face-connected to one of its OWN seeds.
    """
    vols = np.zeros(len(seeds))
    for i, sd in enumerate(seeds):
        m = labels == i
        if not m.any():
            continue
        cc, _ = ndi.label(m, structure=conn)
        keep = {int(cc[s]) for s in sd if cc[s] > 0}
        good = np.isin(cc, list(keep)) if keep else np.zeros_like(m)
        labels[m & ~good] = -1
        vols[i] = float(good.sum())
    return vols

def _shell_volumes(labels, stack, w, holes, k):
    """Volume that COUNTS toward the target.
    For a solid structure that is its whole territory. For a hollow one like a vessel the
    lumen is inside the territory and is claimed, so nothing leaks into it, but it is NOT
    tissue and must count zero."""
    m = labels == k
    if holes is None or holes[k] <= 0.0:
        return float(m.sum())
    return float((m & (stack[k] > holes[k] * w[k])).sum())

def _assign(stack, w, seeds, conn, mask, holes=None):
    val = stack - w.reshape((-1,) + (1,) * (stack.ndim - 1)).astype(np.float32)
    arg = np.argmin(val, axis=0).astype(np.int16)
    best = np.take_along_axis(val, arg[None], axis=0)[0]
    labels = np.where(best < 0, arg, np.int16(-1)).astype(np.int16)
    if mask is not None:
        labels[~mask] = -1
    _prune_to_seeds(labels, seeds, conn)          # the LUMEN stays part of the body
    vols = np.array([_shell_volumes(labels, stack, w, holes, k) for k in range(len(seeds))])
    return labels, vols


def _grow_to_targets(labels, vols, stack, w, targets, conn, mask):
    """Greedy priority growth until every structure hits its target.
    Prune Voxels that got separated from their seed structure"""
    need = {i: int(round(targets[i] - vols[i])) for i in range(len(targets))
            if targets[i] - vols[i] >= 1}
    if not need:
        return labels, vols
    shape = labels.shape
    flat = labels.reshape(-1)
    heap = []
    for i in need:
        m = labels == i
        fr = ndi.binary_dilation(m, structure=conn) & ~m & (labels == -1)
        if mask is not None:
            fr &= mask
        p = np.flatnonzero(fr.ravel())
        key = (stack[i].ravel()[p] - w[i]).astype(float)
        heap.extend(zip(key.tolist(), [i] * p.size, p.tolist()))
    heapq.heapify(heap)
    steps = [int(np.prod(shape[a + 1:])) for a in range(len(shape))]
    while heap and need:
        _, i, p = heapq.heappop(heap)
        if i not in need or flat[p] != -1:
            continue
        flat[p] = i
        vols[i] += 1
        need[i] -= 1
        if need[i] <= 0:
            del need[i]
            continue
        coord = np.unravel_index(p, shape)
        for a, s in enumerate(steps):
            for d in (-1, 1):
                c = coord[a] + d
                if not (0 <= c < shape[a]):
                    continue
                q = p + d * s
                if flat[q] == -1 and (mask is None or mask.reshape(-1)[q]):
                    heapq.heappush(heap, (float(stack[i].reshape(-1)[q] - w[i]), i, int(q)))
    return labels, vols


def carve_to_coverage(labels, coverage, score_extra=None, margin=2.0, hole=None):
    """Remove random nematic field until tissue occupies exactly round(coverage*N) voxels.

    Voxels are ranked by distance from the nearest structure (plus whatever `score_extra`
    adds), and the worst-ranked are dropped.
    The margin guarantees any voxel within `margin` of
    a structure is unconditionally kept, so every structure keeps a stromal rim.

    Returns (tissue, dist, score). Raises if the structures plus their collars already
    exceed the coverage budget, rather than silently eating the collar.
    """
    shape = labels.shape
    n_tot = int(np.prod(shape))
    n_keep = int(round(float(coverage) * n_tot))
    struct = labels >= 0
    hole = np.zeros(shape, bool) if hole is None else np.asarray(hole, bool)
    solid = struct & ~hole
    dist = ndi.distance_transform_edt(~struct)
    score = dist.astype(np.float64)
    if score_extra is not None:
        score = score + np.asarray(score_extra, np.float64)
    protected = solid | ((dist <= float(margin)) & ~hole)
    n_prot = int(protected.sum())
    if n_prot > n_keep:
        raise ValueError(
            f"infeasible: the structures plus a {margin:g}-voxel collar need {n_prot} "
            f"voxels ({n_prot / n_tot:.1%} of the frame) but COVER allows {n_keep} "
            f"({coverage:.1%}). Lower a FRAC, lower MARGIN_UM, or raise COVER.")
    if n_keep > n_tot - int(hole.sum()):
        raise ValueError(
            f"infeasible: COVER asks for {n_keep} tissue voxels but {int(hole.sum())} are "
            f"lumen, leaving only {n_tot - int(hole.sum())}. Lower COVER, lower a vessel's "
            f"FRAC, or raise WALL_IN so its lumen is smaller.")
    score[protected] = -np.inf
    score[hole] = np.inf                       # a lumen is never tissue
    keep = np.argpartition(score.ravel(), n_keep - 1)[:n_keep]
    tissue = np.zeros(n_tot, bool)
    tissue[keep] = True
    return tissue.reshape(shape), dist, score


def pressure_pack(shape, seeds, targets, distances, mask=None, holes=None, damping=0.9,
                  tol=2e-3, max_iter=250, stall_patience=30):
    """Inflate each structure until it occupies exactly `targets[i]` voxels.

    Returns (labels, weights, volumes, iterations, converged)."""
    ndim = len(shape)
    conn = ndi.generate_binary_structure(ndim, 1)
    targets = np.asarray(targets, float)
    stack = np.stack([np.asarray(d, np.float32) for d in distances])
    kk = np.array([max(len(s), 1) for s in seeds], float)

    # start at the free-space radius: the answer if the structure had no neighbours
    ball = np.pi ** (ndim / 2.0) / gamma(ndim / 2.0 + 1.0)
    w = (np.maximum(targets, 1.0) / ball) ** (1.0 / ndim)
    cap = 0.25 * w + 1.0
    n_tot = float(np.prod(shape))
    labels = np.full(shape, -1, np.int16)
    vols = np.zeros(len(targets))
    best, last, it, converged = np.inf, 0, 0, False

    for it in range(1, max_iter + 1):
        labels, vols = _assign(stack, w, seeds, conn, mask, holes)
        err = float(np.abs(targets - vols).max())
        if err / n_tot < tol:
            converged = True
            break
        # dV/dw is the interface AREA, so dividing the volume error by it turns the error
        # into a step in RADIUS units
        surf = np.maximum(ndim * ball ** (1.0 / ndim)
                          * np.maximum(vols, 1.0) ** ((ndim - 1.0) / ndim)
                          * kk ** (1.0 / ndim), 6.0)
        w += np.clip(damping * (targets - vols) / surf, -cap, cap)
        if err < best - 5e-4 * n_tot:
            best, last = err, it
        elif it - last > stall_patience:
            break
    for _ in range(40):
        over = vols > targets
        if not over.any():
            break
        w[over] -= np.maximum(0.35, 0.015 * w[over])
        labels, vols = _assign(stack, w, seeds, conn, mask, holes)

    labels, vols = _grow_to_targets(labels, vols, stack, w, targets, conn, mask)
    return labels, w, vols, it, converged

# def build_architecture(tape, ARCH, TG, shape, spacing, um_per_vox, L=4, l_min=2):
#     """Field -> seeds -> pressurised expansion -> carve. 
#     Returns (support, field, report)."""
#     nz, ny, nx = shape
#     n_tot = int(nz) * int(ny) * int(nx)
#     cover = float(np.clip(TG.COVER.v, 0.0, 1.0))


# build_architecture(tape, ARCH, TG, shape, spacing, um_per_vox,
#                                                    L=L, l_min=l_min)

In [ ]:
import sys; sys.path.insert(0, "../src")      # adjust to your layout
import numpy as np, matplotlib.pyplot as plt
from scipy import ndimage as ndi

from scipy import ndimage as ndi
rng=np.random.default_rng(11); shape=(256,256); N=256*256
psi=ndi.gaussian_filter(rng.standard_normal(shape),30)
gy,gx=np.gradient(psi); u=np.stack([-gy,gx]); u/=np.maximum(np.linalg.norm(u,axis=0),1e-9)
u=u.astype(np.float32)
seeds=place_seeds(shape,[2,3],rng.random((400,2)),spread=.9)
d0=distance_uniform(shape,seeds[0],[0.3,0.95],5.0)
d1=distance_field(shape,seeds[1],2.0,director=u,downsample=1)
tg=[0.16*N,0.26*N]
lab,w,vols,it,conv=pressure_pack(shape,seeds,tg,[d0,d1])
tis,dist,score=carve_to_coverage(lab,0.82,margin=0.0)
fig,ax=plt.subplots(1,4,figsize=(16,4.2))
ax[0].imshow(np.where(lab<0,np.nan,lab),cmap="Set2")
ax[0].set_title("labels  %.4f / %.4f  (want .1600 / .2600)"%(vols[0]/N,vols[1]/N),fontsize=9)
ax[1].imshow(score,cmap="viridis"); ax[1].set_title("carve score = distance from structures",fontsize=9)
ax[2].imshow(tis,cmap="gray"); ax[2].set_title("tissue: %.4f (want 0.6200)"%tis.mean(),fontsize=9)
S4=ndi.generate_binary_structure(2,1)
contacts=int((ndi.binary_dilation(lab>=0,structure=S4)&~tis).sum())
comp=np.zeros(shape+(3,)); comp[...,0]=tis*.25
comp[lab==0]=[.88,.64,.24]; comp[lab==1]=[.27,.75,.68]
ax[3].imshow(comp); ax[3].set_title("collar intact -- struct/bg contacts = %d"%contacts,fontsize=9)
for a_ in ax: a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show(); plt.savefig("/tmp/cell1011.png", dpi=70)

In [ ]:
shape = (160, 160); seed = [(80, 40)]; asp = 6.0                                                                                                                                                                  
curv = 0.35                       # degrees turned per pixel of x -- the "how much it bends" knob                                                                                                                  
                                                                                                                                                                                                                    
Y, X = np.mgrid[0:160, 0:160].astype(np.float64)                                                                                                                                                                   
base = np.array([-1.0, 0.4])                          # the direction at the CENTRE of the frame                                                                                                                   
base /= np.linalg.norm(base)                                                                                                                                                                                       
a = np.deg2rad(curv) * (X - 100)                       # linear turn, zero at the centre                                                                                                                           
ca, sa = np.cos(a), np.sin(a)                                                                                                                                                                                      
u = np.stack([base[0] * ca - base[1] * sa, base[0] * sa + base[1] * ca])                                                                                                                                           
                                                                                                                                                                                                                    
d = distance_field(shape, seed, asp, director=u.astype(np.float32), downsample=1)                                                                                                                               
w = 130

Y2, X2 = np.mgrid[0:160:8, 0:160:8]                                                                                                                                                                                
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(d, cmap="magma"); ax[0].contour(d, levels=[w], colors="w")
ax[0].set_title("curved growth (d)")
mask = (d <= w)[::8, ::8]
ax[1].imshow(d, cmap="magma")
ax[1].quiver(X2[mask], Y2[mask], u[1][::8, ::8][mask], u[0][::8, ::8][mask],
    headwidth=0, headlength=0, headaxislength=0, pivot="mid", scale=25, color="cyan")
ax[1].set_title("wall wraps the curve")
for a_ in ax: a_.set_aspect(1); a_.invert_yaxis(); a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
shape = (160, 160); seed = [(80, 80)]; asp = 4.0

# 1. growth flow -- curved, swirly (same as cell 07)
rng = np.random.default_rng(1)
psi = ndi.gaussian_filter(rng.standard_normal(shape), 50)
gy, gx = np.gradient(psi)
u_growth = np.stack([-gy, gx])
u_growth /= np.maximum(np.linalg.norm(u_growth, axis=0), 1e-9)

# 2. grow the tube along it -- a snaking, elongated territory
d = distance_field(shape, seed, asp, director=u_growth.astype(np.float32), downsample=15)

# 3. the wall field: perpendicular to d's OWN gradient, at every point
gdy, gdx = np.gradient(d)
er = np.stack([gdy, gdx]); er /= np.maximum(np.linalg.norm(er, axis=0), 1e-9)   # radial
ec = np.stack([-er[1], er[0]])                                                  # rotate 90° -- wraps the wall

w = 40                             # a pressure to grow to
band = np.abs(d - w) < 8           # draw glyphs only near that wall, not everywhere

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(d, cmap="magma"); ax[0].contour(d, levels=[w], colors="w")
ax[0].set_title("curved growth (d)")
Y, X = np.mgrid[0:160:5, 0:160:5]
mask = band[::5, ::5]
print(X[mask].shape)
print(ec[1][::5, ::5][mask].shape)
ax[1].imshow(d, cmap="magma"); #ax[1].contour(d, levels=[w], colors="w")
ax[1].quiver(X[mask], Y[mask], ec[1][::5, ::5][mask], ec[0][::5, ::5][mask],
    headwidth=0, headlength=0, headaxislength=0, pivot="mid", scale=25, color="cyan")
ax[1].set_title("wall wraps the curve")
for a_ in ax: a_.set_aspect(1); a_.invert_yaxis(); a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
from skimage import measure
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# 3D Distance field
shape_3d = (60, 60, 60)
seeds_3d = [(30, 30, 30)]
axis_3d = (0.0, 0.0, 1.)
d_3d = distance_uniform(shape_3d, seeds_3d, axis_3d, aspect=0.7)

# Extract isosurface at distance = 15.0
verts, faces, _, _ = measure.marching_cubes(d_3d, level=15.0)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
mesh = Poly3DCollection(verts[faces], alpha=0.7, edgecolor='k', linewidths=0.2)
mesh.set_facecolor('royalblue')
ax.add_collection3d(mesh)

ax.set_xlim(0, 60)
ax.set_ylim(0, 60)
ax.set_zlim(0, 60)
ax.set_title("3D Isosurface ($d(x) = 15$)")
plt.show()

In [ ]:
# For now, just two predesigned cell types
def _marker(name, dye, comps, amp=2.0):
    return PanelMarker(name=name, fluorophore=dye, amp=amp, polarity=0.0,
                       noise_components=comps)

PANEL = MarkerPanel(Markers=dict(
    DAPI=dapi_marker(),
    r=_marker("r", "APC", [ClusterNoise(w=.8, s=1.4, mu=.55, width=.45, sharp=5.,
                                        scale=.35, clust=2.0, fill=.22, soft=.25),
                           FibreNoise(w=.3, s=1.3, mu=.2, width=.7, sharp=4.,
                                      lam=.25, length=6.)], amp=2.4),
    g=_marker("g", "FITC", [BlobNoise(w=.5, s=1.5, mu=.1, width=.6, sharp=7.5, scale=.45),
                            NetworkNoise(w=.9, s=1.4, mu=.2, width=1.1, sharp=7.,
                                         scale=1.0, coherence=.6)], amp=1.2)
))


TISSUE_CELL = CellGeometry(RADIUS=8.0, ROUGH=0.15, ELONG=1.4, NUC_FRAC=0.40, RIM=0.0,
                           NUC_OFFSET=0.8)

def _expr(**levels):
    return {k: P(0., 2., .05, float(v), f"expr {k}",
                 comment="expression level; 0 = negative, 1 = the panel's nominal brightness")
            for k, v in levels.items()}

# stroma is r-high / b-mid and g-negative; tumour is the other way round. Same textures --
# only the levels differ, which is exactly how a real panel separates cell types.
CELLTYPES = {
    "stroma": CellType(name="stroma", Color="red", Geometry=TISSUE_CELL,
                       Expression=_expr(DAPI=1.0, r=1.2, b=0.9)),
    "tumour": CellType(name="tumour", Color="blue", Geometry=TISSUE_CELL,
                       Expression=_expr(DAPI=1.0, g=1.4, b=0.6, r=0.15)),
}
FRACTIONS = [0.7, 0.3]
T_POOLS = PANEL.n_pools()
tape.drawTissue(shape=TISSUE_VOL, n_cand=N_CAND, Pool=T_POOLS)

TG = TissueGeometry()

In [ ]:
t_img, labels, nuc_labels, tau_img, types, info = build_tissue(
    tape=tape, TG=TG, shape=TISSUE_VOL, base_geom=TISSUE_CELL, spacing=SPACING,
    Panel=PANEL, CellTypes=CELLTYPES, Fractions=FRACTIONS, um_per_vox=UM_PER_VOX,
    L=L, l_min=L_MIN)
t_rgb = to_rgb(labels > 0, t_img)

In [ ]:
cmap = {name: ct.Color for name, ct in CELLTYPES.items()}
# PLot tissue mask
mask = info['support'][0].astype(int)
cv = np.array([info["centres_vox"][n] for n in info["labels_present"]])
for i in range(info['support'].shape[0]-1):
    mask += info['support'][i].astype(int)
    plt.scatter(cv[i,0], cv[i,1], c = cmap[types[i+1]])
plt.imshow(mask)

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)
tape.drawSensor(shape=(160, 160, 3))

subs = {marker: kryostat(v, optics) for marker, v in t_img.items()}   # keep BOTH vol and z

# We can just use some cell type here as they all use the same channel names
t_psf = np.stack([psf_project(v, z, optics, PANEL.Markers[name].fluorophore)
                  for name, (v, z) in subs.items()], -1)
t_markers = list(subs)
img_adu = detector(t_psf, PANEL, DETECTOR, optics, tape, t_markers)

In [ ]:
sub, sub_z = kryostat(labels, optics)

# plt.imshow(np.sum(sub, axis=0))
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[1])# , cmap = "Purples_r"
plt.axis('off')
plt.show()
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0])# , cmap = "Purples"
plt.axis('off')
plt.show()

In [ ]:
sub, sub_z = kryostat(labels, optics)
sub_n, sub_n_z = kryostat(nuc_labels, optics)


plt.imshow(NormalizeData(img_adu))
# plt.contour(sub[0], colors="red" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green
# " , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99, keep_largest=False)[0], colors="yellow" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
# img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

# # `lbl`/`lbl_z` (and the nucleus pair) are passed through interact's `fixed(...)` below, so the
# # callback binds the tissue label slab captured when THIS cell runs. Without that it would read
# # the module-level `sub`, which Section 3's load cell reassigns to a 4D frame batch -- feeding a
# # 4D array into mask_collapse then crashes fftconvolve with a dimensionality mismatch.
# def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
#                  g_low=0.0, g_high=1.0, g_gamma=1.0,
#                  b_low=0.0, b_high=1.0, b_gamma=1.0,
#                  show_masks=True,
#                  lbl=None, lbl_z=None, lbl_n=None, lbl_n_z=None):
    
#     adjusted_img = np.zeros_like(img_psf_norm)
#     params = [(r_low, r_high, r_gamma), 
#               (g_low, g_high, g_gamma), 
#               (b_low, b_high, b_gamma)]
    
#     for i, (low, high, gamma) in enumerate(params):
#         c_data = img_psf_norm[:, :, i]
#         denom = max(high - low, 1e-6)
#         c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
#         adjusted_img[:, :, i] = c_norm ** gamma

#     plt.figure(figsize=(8, 8))
#     plt.imshow(adjusted_img)
    
#     # Checkbox logic
#     if show_masks:
#         # plt.contour(lbl.any(0), colors="red" , origin="lower", alpha=.55)
#         # plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
#         plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
#         # plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
#         # plt.contour(lbl.any(0), colors="red" , origin="lower", alpha=.55)
#         # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
#         # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
#         # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        
#     plt.axis('off')
#     plt.show()

# interact(update_image, 
#          r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
#          r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
#          r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
#          # Repeat for G and B...
#          g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
#          g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
#          g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
#          b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
#          b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
#          b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
#          show_masks=Checkbox(value=True, description='Show Masks'),
#          # bind the tissue slab now, so a later `sub` reassignment can't reach this callback
#          lbl=fixed(sub), lbl_z=fixed(sub_z),
#          lbl_n=fixed(sub_n), lbl_n_z=fixed(sub_n_z)
# )
# print("done")

## 2.1) Introducing Artifacts

In [ ]:
from parameter import Artifacts, ArtifactMap, Fussel, Aggregate, ArtifactMask, Detachment
import artifacts as ART

In [ ]:
AR = Artifacts(
    Map=ArtifactMap(MIN_DIST=12.0, COVER=0.60),
    Fussel=Fussel(N=0, WIDTH_UM=2.5, WOBBLE_UM=7.0),
    Aggregate=Aggregate(N=3, DIAM_UM=1.2, GAIN=120.0),
    Mask=ArtifactMask(DILATE_PX=2.0),
    Detachment=Detachment(ELEVATION_UM=6.0, COVER=0.10, EDGE_BIAS=5., EDGE_WIDTH_UM=7.0),
)

tape.drawArtifacts(shape=TISSUE_VOL, **cfg.ARTIFACTS)

In [ ]:
t_img_art = {k: v.copy() for k, v in t_img.items()}
art = build_artifacts(vols = t_img_art, tape = tape, AR = AR, opt = optics, shape = TISSUE_VOL, panel = PANEL, um_per_vox = UM_PER_VOX, spacing = SPACING, geom=TISSUE_CELL, tissue_support=info["support"], L=L, l_min=L_MIN)


DETACH = ART.detachment_map(tape, AR.Detachment, info["support"], SPACING, UM_PER_VOX, optics.um_per_pz)
SHIFT  = DETACH['shift']
N_EXT  = int(SHIFT.max())
FLAT   = np.zeros_like(SHIFT)


def render_adu(vols, shift=SHIFT, n_ext=N_EXT):
    """kryostat -> displace the cut slab -> psf_project -> detector.
    """
    ss = {m: kryostat(v, optics) for m, v in vols.items()}
    psf = np.stack([ART.lift_project(v, z, shift, n_ext, optics, PANEL.Markers[n].fluorophore)
                    for n, (v, z) in ss.items()], -1)
    return detector(psf, PANEL, DETECTOR, optics, tape, list(ss))

fig, ax = plt.subplots(1, 3, figsize = (16, 8))
adu_art = render_adu(t_img_art)
adu = render_adu(t_img, shift=FLAT, n_ext=0)
sub_art, z_art = kryostat(art["labels"], optics)
sub_art, z_art = ART.lift_slab(sub_art, SHIFT, N_EXT), ART.extend_z(z_art, N_EXT, optics.um_per_pz)
sub_lift, sub_lift_z = ART.lift_slab(sub, SHIFT, N_EXT), ART.extend_z(sub_z, N_EXT, optics.um_per_pz)

ax[0].imshow(NormalizeData(adu_art))
# ax[0].contour(mask_collapse(sub_art, z_art, optics, mask_pct=0.90)[0], colors="violet", origin="lower", alpha=1.)
# ax[0].contour(DETACH["lifted"], colors="orange", origin="lower", alpha=.9)
ax[0].set_title("artifacts (violet = debris, orange = detached)")
ax[1].imshow(NormalizeData(adu))
ax[1].set_title("clean: flat section, no artifacts")
im = ax[2].imshow(DETACH["elevation_um"], cmap="magma")
ax[2].set_title("elevation off the slide (um)")
plt.colorbar(im, ax=ax[2], fraction=0.046)
plt.show()

In [ ]:
img_psf_norm = NormalizeData(np.maximum(adu_art - DETECTOR.OFFSET_ADU.v, 0))

# `lbl`/`lbl_z` (and the nucleus pair) are passed through interact's `fixed(...)` below, so the
# callback binds the tissue label slab captured when THIS cell runs. Without that it would read
# the module-level `sub`, which Section 3's load cell reassigns to a 4D frame batch -- feeding a
# 4D array into mask_collapse then crashes fftconvolve with a dimensionality mismatch.
def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0,
                 show_masks=True, show_art_masks=True,
                 lbl=None, lbl_z=None, sub_art=None, z_art=None):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    
    # Checkbox logic
    if show_masks:
        plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        
    if show_art_masks:
        plt.contour(mask_collapse(sub_art, z_art, optics, mask_pct=0.90)[0], colors="violet" , origin="lower", alpha=.55)
        plt.contour(DETACH["lifted"], colors="orange", origin="lower", alpha=.9)
        
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
         show_masks=Checkbox(value=True, description='Show Masks'),
         show_art_masks=Checkbox(value=True, description='Show Artifact Masks'),
         # bind the tissue slab now, so a later `sub` reassignment can't reach this callback
         lbl=fixed(sub), lbl_z=fixed(sub_z),
         sub_art=fixed(sub_art), z_art=fixed(z_art)
)
print("done")

# 3) Inspect a generated dataset

In [ ]:
DATA = Path("../data/example")

_stem = str(DATA.resolve())
if not os.path.exists(_stem + ".npy"):
    raise FileNotFoundError(
        f"no dataset at {_stem}.npy -- generate one first:\n"
        f"    python src/gen_dataset.py make --mode tissue --n 500 --workers 8 --out {_stem}")

meta = np.load(_stem + ".meta.npz", allow_pickle=True)
imgs = np.load(_stem + ".npy", mmap_mode="r")

theta = meta["theta"]
paths = [str(p) for p in meta["paths"]]
n, H, W, C = imgs.shape
# `names`/`dyes` are per channel in image order
names = [str(x) for x in meta["names"]] if "names" in meta.files else \
        [cfg.DAPI_NAME] + [f"ch{k}" for k in range(1, C)]
print(meta["dyes"])
dyes = [str(x) for x in meta["dyes"]] if "dyes" in meta.files else None

print(f"{_stem}")
print(f"  {n} frames, {H}x{W} px, {C} channels, {imgs.dtype} seed={meta['seed']}")
print(f"  {os.path.getsize(_stem + '.npy') / 1e9:.3f} GB on disk, memmapped -- resident cost "
      f"is one frame ({H * W * C * 2 / 1e6:.2f} MB), not the file")
print(f"  theta {theta.shape}: {len(paths)} fitted parameters, normalised to [0, 1]")
print("  channels: " + ", ".join(f"{k}:{nm}" + (f"/{dyes[k]}" if dyes else "")
                                 for k, nm in enumerate(names)))

rng = np.random.default_rng(0)
pick = np.sort(rng.choice(n, size=min(n, 64), replace=False))
sample = np.asarray(imgs[pick], np.float32)

print(f"\nper-channel ADU over {len(pick)} sampled frames "
      f"(black level = DETECTOR OFFSET_ADU = {cfg.DETECTOR['OFFSET_ADU']:.0f})")
print(f"  {'channel':>10}  {'median':>8} {'p99':>8} {'max':>8}   {'at ceiling':>10}")
for k, nm in enumerate(names):
    c = sample[..., k]
    print(f"  {nm:>10}  {np.median(c):8.0f} {np.percentile(c, 99):8.0f} {c.max():8.0f}"
          f"   {100 * (c >= cfg.ADU_MAX).mean():9.3f}%")
    if np.percentile(c, 99) < cfg.DETECTOR["OFFSET_ADU"] + 20:
        print(f"{'':14}^ essentially dark: no cell type expresses {nm}")

In [ ]:
def stretch(frame, p=(1.0, 99.5)):
    
    a = np.asarray(frame, np.float32)
    out = np.zeros(a.shape, np.float32)
    for k in range(a.shape[-1]):
        v0, v1 = np.percentile(a[..., k], p)
        out[..., k] = np.clip((a[..., k] - v0) / max(v1 - v0, 1e-6), 0.0, 1.0)
    return out


def as_rgb(frame):
    """First three channels as RGB, with channel 0 (always DAPI) in BLUE by convention."""
    a = stretch(frame)
    rgb = np.zeros(a.shape[:2] + (3,), np.float32)
    for k in range(min(3, a.shape[-1])):
        rgb[..., 2 - k] = a[..., k]        # ch0 -> blue, ch1 -> green, ch2 -> red
    return rgb


nrow, ncol = 3, 4
show = pick[:nrow * ncol]
fig, axes = plt.subplots(nrow, ncol, figsize=(2.7 * ncol, 2.7 * nrow))
for ax, i in zip(np.ravel(axes), show):
    ax.imshow(as_rgb(imgs[i]))            # one frame off disk, per panel
    ax.set_title(f"#{i}", fontsize=8)
    ax.axis("off")
for ax in np.ravel(axes)[len(show):]:
    ax.axis("off")
_lbl = " / ".join(f"{nm}={c}" for nm, c in zip(names[:3], ("blue", "green", "red")))
fig.suptitle(f"{len(show)} of {n} frames   ({_lbl})", fontsize=10)
plt.tight_layout()
plt.show()

# Every channel of a single frame. This is the one to compare against the notebook's own
# tissue render above -- same forward model, so the textures should be recognisably the same.
i0 = int(show[0])
fig, axes = plt.subplots(1, C, figsize=(3.0 * C, 3.4))
for k, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(stretch(imgs[i0])[..., k], cmap="gray")
    ax.set_title(names[k] + (f"  ({dyes[k]})" if dyes else ""), fontsize=9)
    ax.axis("off")
fig.suptitle(f"frame #{i0}, all {C} channels, 1-99.5 percentile stretch", fontsize=10)
plt.tight_layout()
plt.show()